# Enterprise Knowledge Assistant — v7

**Version:** v7  
**Status:** POC — production candidate  
**Stack:** Pure Python · BM25 · No API keys · No LLM · No cloud  

---

**Total KB:** 50 documents (was 42)  

---

## Cell 0 — Install dependencies

Run once on first use. Skip if already installed.

In [1]:
# Install required packages
# Run this cell once, then restart kernel if prompted
import subprocess, sys

packages = [
    'rank-bm25',
    'python-docx',
    'openpyxl',
    'python-pptx',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

print('\nAll dependencies installed. Ready to run Cell 1.')

  ✓ rank-bm25
  ✓ python-docx
  ✓ openpyxl
  ✓ python-pptx

All dependencies installed. Ready to run Cell 1.


## Cell 1 — Initialise the assistant

Runs the full startup pipeline:
1. Load manual synonym registry  
2. Ingest all 50 documents  
3. Auto-synonym discovery  
4. BM25 index build  

Expected: ~10–20 seconds depending on hardware.

In [2]:
import sys, os

# Add engine directory to path
ENGINE_DIR = os.path.join(os.getcwd(), 'engine')
if ENGINE_DIR not in sys.path:
    sys.path.insert(0, ENGINE_DIR)

from knowledge_assistant import EnterpriseKnowledgeAssistant

assistant = EnterpriseKnowledgeAssistant(
    knowledge_base_dir = './knowledge_base',
    outputs_dir        = './outputs',
)

2026-05-18 11:37:15,527 - INFO - 
2026-05-18 11:37:15,528 - INFO - ENTERPRISE KNOWLEDGE ASSISTANT v7
2026-05-18 11:37:15,529 - INFO - 4 Fixes: Chunk Dedup | Capacity Intent | LIST Risk | Param Noise
2026-05-18 11:37:15,530 - INFO - =================================================================
2026-05-18 11:37:15,532 - INFO -   Manual synonym registry: 43 groups, 222 aliases
2026-05-18 11:37:15,532 - INFO - =================================================================
2026-05-18 11:37:15,533 - INFO - PHASE 1: Ingestion Engine v7
2026-05-18 11:37:15,534 - INFO -   Base: knowledge_base
2026-05-18 11:37:15,535 - INFO - =================================================================
2026-05-18 11:37:15,541 - INFO -   Found 50 files: .docx:6, .pptx:5, .txt:34, .xlsx:5
2026-05-18 11:37:15,543 - INFO -   [.txt ] [application_parameters] kafka_messaging_guide.txt — 2,438ch  5 chunks  quality:structured
2026-05-18 11:37:15,545 - INFO -   [.txt ] [application_parameters] parameter_refer

## Cell 2 — v7 Validation Queries

These four queries directly validate the four v7 fixes.

**Expected outcomes:**
- Fix 1 (dedup): No repeated content blocks in the weekend handover or ETL troubleshoot outputs
- Fix 2 (capacity intent): Intent line shows `ANALYTICAL`, not `INCIDENT`
- Fix 3 (LIST risk): Redis TTL, null rate gate, FRAUD_KAFKA_CONSUMER_GROUP appear in output
- Fix 4 (param noise): Query suggestions do not mention WEEKEND or SYMPTOMS

In [3]:
# ── Fix 1 Validation: Chunk deduplication ─────────────────────────────────
# Previously: ISSUE: Reporting Service slow appeared 3x in output
# Expected: each section appears exactly once
print('=' * 70)
print('FIX 1 VALIDATION — Chunk deduplication')
print('Check: no repeated content blocks in output')
print('=' * 70)
_ = assistant.print_answer(
    'Walk me through the ETL job failure troubleshooting steps and what causes it'
)

2026-05-18 11:37:26,639 - INFO - 
  Query: "Walk me through the ETL job failure troubleshooting steps and what causes it"
2026-05-18 11:37:26,667 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (93%)
2026-05-18 11:37:26,675 - INFO -   Output: 20260518_113726_summary_high_walk_me_through_the_etl_job_failure_troubleshootin.txt


FIX 1 VALIDATION — Chunk deduplication
Check: no repeated content blocks in output
SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Operations Troubleshooting Runbook  [.TXT]
  Topic  : Troubleshooting, Incident Response, Common Errors
  Owner  : Platform Engineering Team  |  v6.2  |  Updated: 2025-05-17

  [ISSUE: Reporting Service slow — timeout on report generation]
    STEPS:
    1. Check DWH query performance — review query plan in Snowflake console.
    2. Verify ETL-001 completed — stale DWH data causes report retries.
    3. Check resource usage: ssh 10.10.1.40, run top or htop.
    4. Check if CBS_BATCH_WINDOW_START overlaps with RPT-001 start time (I/O contention).
    ESCALATE: bi-ops@company.com

  [ISSUE: ETL-001 job failed — DWH showing stale data]
    STEPS:
    1. Check Airflow at http://10.10.50.30:8081 — find dag_cbs_to_dwh, review task logs.
    2. Common cause 1: EOD-001 did not complete. Check if

In [4]:
# ── Fix 2 Validation: Capacity breach → ANALYTICAL ────────────────────────
# Previously: routed to INCIDENT (bare `breach` matched INCIDENT pattern)
# Expected: Intent line shows ANALYTICAL
print('=' * 70)
print('FIX 2 VALIDATION — Capacity breach intent')
print('Expected: Intent = ANALYTICAL (not INCIDENT)')
print('=' * 70)
_ = assistant.print_answer(
    'Which system is closest to a capacity breach?'
)

2026-05-18 11:37:42,937 - INFO - 
  Query: "Which system is closest to a capacity breach?"
2026-05-18 11:37:42,949 - INFO -   Intent: ANALYTICAL  |  Confidence: HIGH (93%)
2026-05-18 11:37:42,966 - INFO -   Output: 20260518_113742_analytical_high_which_system_is_closest_to_a_capacity_breach.txt


FIX 2 VALIDATION — Capacity breach intent
Expected: Intent = ANALYTICAL (not INCIDENT)
ANALYTICAL BRIEF
─────────────────────────────────────────────────────────────────
Query: Which system is closest to a capacity breach?

FINDINGS BY SOURCE
────────────────────────────────────────

[1] Service Level Agreement Register and Actuals  [.TXT]  [sla_contracts]
  Owner: Platform Engineering + Business Relationship Management  |  Updated: 2025-04-15

  [SHEET 3 — BREACH DETAIL LOG]
    BREACH: INC-2024-1198
    Date: 2024-12-14 02:17–04:35 (138 min total)
    Affected: FRAUD-ENGINE (primary), PAYMENTS-GW (cascade), CBS (partial degradation)
    Root Cause: Fraud Engine Redis ran Out of Memory (OOM). FRAUD_REDIS_TTL_SEC had been
    reduced from 3600 to 300 by Risk Tech team without Platform Engineering sign-off.
    This caused 12x recompute load, overwhelming Redis memory allocation.
    Resolution: Redis flushed, FRAUD_REDIS_TTL_SEC restored to 3600, OOM alert threshold lowered.
    SLA Im

In [5]:
# ── Fix 3 Validation: LIST risk fallback ──────────────────────────────────
# Previously: LIST only extracted INC-/REL- prefixed lines, missed risk content
# Expected: FRAUD_REDIS_TTL_SEC, FRAUD_KAFKA_CONSUMER_GROUP, null rate gate appear
print('=' * 70)
print('FIX 3 VALIDATION — LIST risk/approval fallback')
print('Expected: Redis TTL, FRAUD_KAFKA_CONSUMER_GROUP, null rate gate in output')
print('=' * 70)
_ = assistant.print_answer(
    'What are all the risks I should know about before approving the fraud engine v3.4.0 release?'
)

2026-05-18 11:37:55,602 - INFO - 
  Query: "What are all the risks I should know about before approving the fraud engine v3.4.0 release?"
2026-05-18 11:37:55,631 - INFO -   Intent: LIST  |  Confidence: HIGH (100%)
2026-05-18 11:37:55,644 - INFO -   Output: 20260518_113755_list_high_what_are_all_the_risks_i_should_know_about_before.txt


FIX 3 VALIDATION — LIST risk/approval fallback
Expected: Redis TTL, FRAUD_KAFKA_CONSUMER_GROUP, null rate gate in output
RESULTS
─────────────────────────────────────────────────────────────────

  [1] [oncall_handover_notes_may2025.txt  .TXT]
    • INC-2025-0044 was caused by this exact thing in january.
    • INC-2025-0288 was filed for this.

─────────────────────────────────────────────────────────────────
ADDITIONAL RISK CONTEXT FROM DOCUMENTS
─────────────────────────────────────────────────────────────────
Query: What are all the risks I should know about before approving the fraud engine v3.4.0 release?

FINDINGS BY SOURCE
────────────────────────────────────────

[1] Oncall Handover Notes May2025  [.TXT]  [unstructured_samples]
  Owner: —  |  Updated: —

  [- redis auto-scaling for fraud engine. sentinel config is done. load testing is ]
    - redis auto-scaling for fraud engine. sentinel config is done. load testing is next week.
    FRAUD_REDIS_TTL_SEC is locked at 3600 unti

In [6]:
# ── Fix 4 Validation: Param extractor noise ───────────────────────────────
# Previously: WEEKEND appeared in query suggestions as if it were a parameter name
# Expected: suggestions reference real parameter names (FRAUD_REDIS_TTL_SEC etc.)
#           and do NOT mention WEEKEND, SYMPTOMS, CONTACTS
print('=' * 70)
print('FIX 4 VALIDATION — Parameter extractor noise')
print('Expected: suggestions show real param names, NOT WEEKEND/SYMPTOMS/CONTACTS')
print('=' * 70)
_ = assistant.print_answer(
    'What should I watch out for this weekend as the on-call engineer?'
)

2026-05-18 11:38:01,004 - INFO - 
  Query: "What should I watch out for this weekend as the on-call engineer?"
2026-05-18 11:38:01,014 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (85%)
2026-05-18 11:38:01,022 - INFO -   Output: 20260518_113801_summary_high_what_should_i_watch_out_for_this_weekend_as_the_on.txt


FIX 4 VALIDATION — Parameter extractor noise
Expected: suggestions show real param names, NOT WEEKEND/SYMPTOMS/CONTACTS
SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Escalation Contacts Adhoc  [.XLSX]
  Topic  : Unstructured Samples
  Owner  : —

  [Section 4]
    Support Portal | Platinum tier
    Compliance / Legal / DPO
    DPO | dpo@company.com | GDPR queries, data subject requests
    Compliance | compliance@company.com
    Legal | legal@company.com
    On-call rotation
    Managed in PagerDuty
    Teams on rotation | Platform Eng, Data Eng, Payments, Security
    Rotation frequency | Weekly
    If no ack in 10 min | Escalates to team lead automatically
    DO NOT contact vendors directly without Platform Engineering sign-off
    All vendor comms must be logged in Servic

  [Section 5]
    t vendors directly without Platform Engineering sign-off
    All vendor comms must be logged in ServiceNow

  [Section 3]

## Cell 3 — Regression Queries

Confirm that v6 fixes still work correctly in v7.

In [7]:
# v6 regression: 401 auth fix — runbook should be [1] with BM25 score >> others
_ = assistant.print_answer('Users are getting 401 errors and cannot log in')

2026-05-18 11:38:25,327 - INFO - 
  Query: "Users are getting 401 errors and cannot log in"
2026-05-18 11:38:25,334 - INFO -   Intent: TROUBLESHOOT  |  Confidence: HIGH (100%)
2026-05-18 11:38:25,343 - INFO -   Output: 20260518_113825_troubleshoot_high_users_are_getting_401_errors_and_cannot_log_in.txt


TROUBLESHOOTING GUIDE
─────────────────────────────────────────────────────────────────

────────────────────────────────────────
[1] Operations Troubleshooting Runbook  [troubleshooting]  (.TXT)

  Section: ISSUE: Users cannot log in — 401 errors — authentication failing — SSO down — login broken
  Root Cause(s):
    • - All apps rejecting requests simultaneously (indicates AUTH-SVC is the root cause)
    • Single user locked    → Check AUTH_MAX_LOGIN_ATTEMPTS in Keycloak admin (account lock after 5 failures).
    • If DOWN: this is the root cause — all apps will return 401 until AUTH-SVC recovers.
    • If unreachable: auth database failure — page Security team immediately.
  Diagnostic Steps:
    FIRST CHECK — is this one user or all users?
    Intermittent for many → Token expiry or refresh issue. Check AUTH_TOKEN_EXPIRY_SEC (default 900s).
    2. Check if PostgreSQL (Keycloak DB) is reachable from AUTH-SVC host:
    Navigate to Users → find affected user → Sessions tab → check if 

In [8]:
# v6 regression: mainframe port lookup
_ = assistant.print_answer('What port does the mainframe use?')

2026-05-18 11:38:44,300 - INFO - 
  Query: "What port does the mainframe use?"
2026-05-18 11:38:44,315 - INFO -   Intent: LOOKUP  |  Confidence: HIGH (93%)
2026-05-18 11:38:44,327 - INFO -   Output: 20260518_113844_lookup_high_what_port_does_the_mainframe_use.txt


ANSWER
─────────────────────────────────────────────────────────────────
[5] port_host_quick_reference.txt › APPLICATION HOST AND PORT REFERENCE
  CBS / Core Banking System / Core Banking / Mainframe / Banking System / Core App:
  Host: 10.10.1.10
  Port: 8080 (HTTP)
  API:  http://10.10.1.10:8080/api/v2
  DB:   Oracle 19c on 10.10.2.10, port 1521
  Tech: Java 17
  PAYMENTS-GW / Payments Gateway / Payment Gateway / Payment Processor / PG:
  Host: 10.10.1.20
  Port: 9000 (HTTP)
  API:  http://10.10.1.20:9000/payments
  DB:   PostgreSQL 15 on 10.10.2.12, port 5432
  Tech: Python 3.11

[1] swift_integration_spec.txt › MESSAGE FLOW
  1. Customer initiates international transfer via CBS.
  2. CBS sends instruction to PAYMENTS-GW (port 9000).
  3. PAYMENTS-GW validates and runs fraud check via FRAUD-ENGINE (port 8500).
  4. If approved, PAYMENTS-GW calls SWIFT-ADAPTER (port 8090).
  5. SWIFT-ADAPTER constructs MT103/MT202 and sends to SWIFT Alliance Gateway (192.168.200.5:3000).
  6. SWIFT G

In [9]:
# v6 regression: CBS 503 cross-source synthesis
# Expected: Slack thread + email thread + runbook all cited
_ = assistant.print_answer('CBS is returning 503, what do I do?')

2026-05-18 11:38:53,565 - INFO - 
  Query: "CBS is returning 503, what do I do?"
2026-05-18 11:38:53,581 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (93%)
2026-05-18 11:38:53,587 - INFO -   Output: 20260518_113853_summary_high_cbs_is_returning_503_what_do_i_do.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Slack Incident Thread Dec2024  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —

  [[09:44] PAGERDUTY_BOT: ALERT - CBS health check failing - http://10.10.1.10:8080]
    [09:44] PAGERDUTY_BOT: ALERT - CBS health check failing - http://10.10.1.10:8080/api/v2/health returning 503

  [[09:47] platform_lead: CBS is returning 503. checking DB connection pool]
    [09:47] platform_lead: CBS is returning 503. checking DB connection pool
    CBS_DB_POOL_SIZE is 50, checking current connections

  [[09:53] dba_team: done. CBS_DB_POOL_SIZE = 80 as emergency.]
    [09:53] dba_team: done. CBS_DB_POOL_SIZE = 80 as emergency.
    give it 2-3 min to take effect as existing connections drain.

  [[10:00] dba_team: permanent fix approved. CBS_DB_POOL_SIZE = 100.]
    [10:00] dba_team: permanent fix approved. CBS_DB_POOL_SIZE = 100.
    adding monitoring alert at 80% pool utilisation

## Cell 4 — New Document Coverage

Queries that exercise the 8 new v7 knowledge base documents.

In [10]:
# New doc: DBA runbook notes — CBS_DB_POOL_SIZE, Redis TTL, Kafka disk
_ = assistant.print_answer('What is the CBS database pool size and why was it changed?')

2026-05-18 11:38:59,142 - INFO - 
  Query: "What is the CBS database pool size and why was it changed?"
2026-05-18 11:38:59,160 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (78%)
2026-05-18 11:38:59,167 - INFO -   Output: 20260518_113859_summary_high_what_is_the_cbs_database_pool_size_and_why_was_it.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Email Thread Engineering Decisions  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —

  [FROM: data eng lead]
    FROM: data eng lead
    TO: platform eng lead, CBS team lead
    SUBJECT: RE: RE: RE: CBS schema change notification process

  [1. any new column added to CBS transaction table must be notified to data eng BE]
    1. any new column added to CBS transaction table must be notified to data eng BEFORE deployment
    2. use the #schema-changes slack channel
    3. data eng gets minimum 3 days notice to update ETL mappings
    4. CBS release checklist now has a mandatory field: "data eng notified Y/N"
    5. if not notified and ETL breaks, CBS team owns the P1 response

  [current consumer groups to never change without coordination:]
    current consumer groups to never change without coordination:
    - fraud-engine-cg-prod (fraud engine consuming fraud.tra

In [11]:
# New doc: Fraud model notes — constraint knowledge
_ = assistant.print_answer(
    'What constraints exist on FRAUD_REDIS_TTL_SEC and who needs to approve changes?'
)

2026-05-18 11:39:12,470 - INFO - 
  Query: "What constraints exist on FRAUD_REDIS_TTL_SEC and who needs to approve changes?"
2026-05-18 11:39:12,481 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (78%)
2026-05-18 11:39:12,490 - INFO -   Output: 20260518_113912_summary_high_what_constraints_exist_on_fraud_redis_ttl_sec_and.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Oncall Handover Notes May2025  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —

  [- redis auto-scaling for fraud engine. sentinel config is done. load testing is ]
    - redis auto-scaling for fraud engine. sentinel config is done. load testing is next week.
    FRAUD_REDIS_TTL_SEC is locked at 3600 until auto-scaling is confirmed. do NOT change this.
    any request from risk tech to reduce TTL needs platform eng co-approval. they know this.

  [SCHEDULED THIS WEEKEND:]
    SCHEDULED THIS WEEKEND:
    - fraud model v3.4.0 is deploying thursday may 8. this is the Q2 quarterly model retrain.
    new model tested at 96.8% accuracy (above 95% threshold). null rate gate is in place now.
    if null rate exceeds 1% in first 15 min post-deploy, pipeline auto-rollbacks to v3.2.1.
    risk tech team is monitoring. dont need to do anything unless pagerduty fires.
    - NO 

In [12]:
# New doc: Network notes — VLAN layout
_ = assistant.print_answer('What VLANs are used in the production network and what are their IP ranges?')

2026-05-18 11:39:18,786 - INFO - 
  Query: "What VLANs are used in the production network and what are their IP ranges?"
2026-05-18 11:39:18,805 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (93%)
2026-05-18 11:39:18,810 - INFO -   Output: 20260518_113918_summary_high_what_vlans_are_used_in_the_production_network_and.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Core Network Architecture Overview  [.TXT]
  Topic  : Network Architecture
  Owner  : Infrastructure Team  |  v3.2  |  Updated: 2024-11-10

  [KEY NETWORK SEGMENTS]
    SEGMENT: PROD-NET-01
    IP Range: 10.10.1.0/24
    Purpose: Production application servers
    VLAN ID: 101
    Gateway: 10.10.1.1
    SEGMENT: PROD-NET-02
    IP Range: 10.10.2.0/24
    Purpose: Production database servers
    VLAN ID: 102
    Gateway: 10.10.2.1
    SEGMENT: MGMT-NET
    IP Range: 10.10.50.0/24
    Purpose: Management and monitoring
    VLAN ID: 150
    Gateway: 10.10.50.1
    SEGMENT: DMZ-NET
    IP Range: 172.16.1.0/24
    Purpose: Externally accessible services
    VLAN ID: 200
    Gateway: 172.16.1.1

  [CORE NETWORK TOPOLOGY]
    The enterprise network is divided into three primary zones:
    1. DMZ (Demilitarized Zone) - External-facing services
    2. Internal Network - Core bu

In [13]:
# New doc: SWIFT compliance notes — cert expiry
_ = assistant.print_answer('When does the SWIFT certificate expire and what is the renewal lead time?')

2026-05-18 11:39:22,372 - INFO - 
  Query: "When does the SWIFT certificate expire and what is the renewal lead time?"
2026-05-18 11:39:22,392 - INFO -   Intent: ANALYTICAL  |  Confidence: HIGH (100%)
2026-05-18 11:39:22,400 - INFO -   Output: 20260518_113922_analytical_high_when_does_the_swift_certificate_expire_and_what_is.txt


ANALYTICAL BRIEF
─────────────────────────────────────────────────────────────────
Query: When does the SWIFT certificate expire and what is the renewal lead time?

FINDINGS BY SOURCE
────────────────────────────────────────

[1] Swift Payments Ops Notes  [.TXT]  [unstructured_samples]
  Owner: —  |  Updated: january 2025

  [COMPLIANCE NOTES]
    COMPLIANCE NOTES
    - SWIFT messages must be archived for 7 years (regulatory requirement)
    - archive location: SWIFT-ADAPTER writes to object storage (S3-compatible, 10.10.50.90)
    - SWIFT_CERT_EXPIRY = 2025-09-30: the SWIFT connectivity cert expires september 2025.
    renewal is lead time 12 weeks (SWIFT HSM hardware). must start by June 2025.
    owner: payments team + network team + security team (joint)
    - SWIFT CSP (Customer Security Programme) annual audit: scheduled october 2025

  [swift and international payments - compliance and ops notes]
    swift and international payments - compliance and ops notes
    author: payment

In [14]:
# New doc: Platform retro — Q2 risk register
_ = assistant.print_answer('What are the top risks identified for Q2 2025?')

2026-05-18 11:39:37,828 - INFO - 
  Query: "What are the top risks identified for Q2 2025?"
2026-05-18 11:39:37,836 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (77%)
2026-05-18 11:39:37,844 - INFO -   Output: 20260518_113937_summary_high_what_are_the_top_risks_identified_for_q2_2025.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Operations Troubleshooting Runbook  [.TXT]
  Topic  : Troubleshooting, Incident Response, Common Errors
  Owner  : Platform Engineering Team  |  v6.2  |  Updated: 2025-05-17

  [ISSUE: Reporting Service slow — timeout on report generation]
    STEPS:
    1. Check DWH query performance — review query plan in Snowflake console.
    2. Verify ETL-001 completed — stale DWH data causes report retries.
    3. Check resource usage: ssh 10.10.1.40, run top or htop.
    4. Check if CBS_BATCH_WINDOW_START overlaps with RPT-001 start time (I/O contention).
    ESCALATE: bi-ops@company.com

  [ISSUE: Fraud Engine high false positives blocking legitimate payments]
    STEPS:
    1. Verify PAYMENTS-GW_FRAUD_THRESHOLD. Default 0.75. Lower value blocks more payments.
    2. Check FRAUD_MODEL_VERSION — if recently updated, model may need recalibration.
    3. Check FRAUD_REDIS_TTL_SEC 

In [15]:
# New doc: New analyst FAQ — basic orientation query
_ = assistant.print_answer(
    'I am a new analyst. What is the difference between CBS and the Payments Gateway?'
)

2026-05-18 11:39:41,015 - INFO - 
  Query: "I am a new analyst. What is the difference between CBS and the Payments Gateway?"
2026-05-18 11:39:41,047 - INFO -   Intent: COMPARISON  |  Confidence: HIGH (100%)
2026-05-18 11:39:41,054 - INFO -   Output: 20260518_113941_comparison_high_i_am_a_new_analyst_what_is_the_difference_between.txt


COMPARISON
─────────────────────────────────────────────────────────────────
Comparing across 5 sources

────────────────────────────────────────
[1] Analyst Faq
     File: new_analyst_faq.txt  (.TXT)  |  Folder: unstructured_samples

  ▸ Q: what's the difference between CBS and Payments Gateway?
      Q: what's the difference between CBS and Payments Gateway?
      A: CBS is the backend account ledger. Payments Gateway (PAYMENTS-GW) is the transaction processor.
    → When a payment happens: Payments GW receives it → calls Fraud Engine for a risk score →
      calls CBS to debit/credit the accounts → returns confirmation.
      CBS doesn't talk to the outside world. Payments GW does.
      PAYMENTS-GW host: 10.10.1.20, Port: 9000.

  ▸ Q: what is the SLA for CBS / payments?
      Q: what is the SLA for CBS / payments?
    → A: CBS SLA: 99.9% availability (target), 99.87% actual Q1 2025 (below target due to INC-2024-1201)
    → Payments GW SLA: 99.95% target, 99.91% actual Q1 2025
    

## Cell 5 — Free-form queries

Run any query here.

In [16]:
# Edit query below and run
_ = assistant.print_answer(
    'your question here',
    top_k=6,
    # folder_filter='troubleshooting'   # optional: restrict to one folder
)

2026-05-18 11:39:56,539 - INFO - 
  Query: "your question here"
2026-05-18 11:39:56,543 - INFO -   Intent: SUMMARY  |  Confidence: HIGH (85%)
2026-05-18 11:39:56,546 - INFO -   Output: 20260518_113956_summary_high_your_question_here.txt


SUMMARY
─────────────────────────────────────────────────────────────────
────────────────────────────────────────
[1] Parameter Notes From Call  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —

  [contact for parameter questions: platform-eng@company.com or raise in #platform-]
    contact for parameter questions: platform-eng@company.com or raise in #platform-engineering slack

────────────────────────────────────────
[2] Analyst Faq  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —  |  Updated: may 2025

  [new analyst / new joiner questions answered - compiled from slack threads]
    new analyst / new joiner questions answered - compiled from slack threads
    compiled by: platform engineering team
    this is a living doc - add questions as they come up
    last updated: may 2025

────────────────────────────────────────
[3] Analyst Question Log  [.TXT]
  Topic  : Unstructured Samples
  Owner  : —

  [ANALYST QUESTION LOG - platform eng slack #ask-platform]
    ANALYST QUES

## Cell 6 — Inspection utilities

In [17]:
# List all ingested documents
import pandas as pd
docs = assistant.get_document_list()
df   = pd.DataFrame(docs)[['filename', 'file_format', 'topic_folder', 'quality', 'char_count']]
print(f'Total: {len(df)} documents')
print(df.to_string(index=False))

2026-05-18 11:40:08,420 - INFO - NumExpr defaulting to 16 threads.


Total: 50 documents
                                    filename file_format           topic_folder      quality  char_count
                   kafka_messaging_guide.txt        .txt application_parameters   structured        2438
               parameter_reference_guide.txt        .txt application_parameters   structured        3431
           architecture_decision_records.txt        .txt architecture_decisions   structured        9149
             platform_technology_roadmap.txt        .txt architecture_decisions   structured        8333
      q1_2025_quarterly_business_review.pptx       .pptx architecture_decisions   structured        5600
                system_capacity_register.txt        .txt      capacity_planning   structured        6520
              api_integration_standards.docx       .docx     connectivity_ports   structured        6596
            application_connectivity_map.txt        .txt     connectivity_ports   structured        2323
               port_host_quick_refe

In [18]:
# Document count by folder and quality
df.groupby(['topic_folder', 'quality']).size().reset_index(name='count')

,topic_folder,quality,count
0,application_parameters,structured,2
1,architecture_decisions,structured,3
2,capacity_planning,structured,1
3,connectivity_ports,structured,5
4,incident_reports,structured,2
5,network_architecture,structured,2
6,process_workflows,structured,6
7,release_management,structured,1
8,sla_contracts,structured,1
9,troubleshooting,structured,2


In [19]:
# View auto-discovered synonyms (not in manual registry)
auto = assistant.get_auto_synonyms()
print(f'Auto-discovered synonym groups: {len(auto)}')
for k, v in list(auto.items())[:20]:
    print(f'  {k:30s} → {v[:4]}')

Auto-discovered synonym groups: 121
  tls                            → ['cross zone']
  transaction status updates     → ['cbs']
  frozen codebases               → ['swift-adapter']
  aws                            → ['cloud', 'dns', 'sqs']
  decisions                      → ['pending arb']
  adr-006                        → ['proposed']
  adr-008                        → ['kraft migration']
  nginx                          → ['payments gw instance behind load balancer']
  hpa                            → ['horizontal pod autoscaler']
  horizontal scale               → ['tps']
  ram                            → ['cpu']
  disk                           → ['this one bites people']
  post                           → ['new resource created']
  udp                            → ['snmp polling']
  application servers            → ['prod-net-01']
  database servers               → ['prod-net-02']
  management                     → ['mgmt-net']
  dmz-net                        → ['external faci

In [20]:
# Session query history
history = assistant.get_query_history()
for h in history:
    print(f"  [{h['intent']:12s}] [{h['confidence']:6s}] {h['query'][:70]}")
    print(f"               → {h['top_source']}")

  [SUMMARY     ] [HIGH  ] Walk me through the ETL job failure troubleshooting steps and what cau
               → operations_runbook.txt
  [ANALYTICAL  ] [HIGH  ] Which system is closest to a capacity breach?
               → sla_register_and_actuals.txt
  [LIST        ] [HIGH  ] What are all the risks I should know about before approving the fraud 
               → oncall_handover_notes_may2025.txt
  [SUMMARY     ] [HIGH  ] What should I watch out for this weekend as the on-call engineer?
               → escalation_contacts_adhoc.xlsx
  [TROUBLESHOOT] [HIGH  ] Users are getting 401 errors and cannot log in
               → operations_runbook.txt
  [LOOKUP      ] [HIGH  ] What port does the mainframe use?
               → swift_integration_spec.txt
  [SUMMARY     ] [HIGH  ] CBS is returning 503, what do I do?
               → slack_incident_thread_dec2024.txt
  [SUMMARY     ] [HIGH  ] What is the CBS database pool size and why was it changed?
               → email_thread_engineering_

In [21]:
# Export session history to JSON
assistant.export_query_history('query_history.json')
print('Exported to query_history.json')

2026-05-18 11:41:11,356 - INFO -   History → query_history.json


Exported to query_history.json


---

## Quick Reference

```python
# Standard query
assistant.print_answer("your question here")

# With top-k control (default 6)
assistant.print_answer("your question", top_k=8)

# Restrict to one folder
assistant.print_answer("your question", folder_filter="troubleshooting")

# Get structured dict (for programmatic use)
r = assistant.ask("your question")
# r.keys(): query, intent, confidence, status, answer, sources, timestamp, output_file

# List all documents
docs = assistant.get_document_list()

# Auto-discovered synonyms only
auto = assistant.get_auto_synonyms()

# Export session history
assistant.export_query_history('query_history.json')
```

